# InsightAid AI — Forecasting Model Training

Trains and evaluates every forecasting candidate (moving average, ARIMA,
Prophet sales-only, Prophet sentiment-enhanced where real sentiment data
exists, XGBoost, and LSTM where the data volume justifies it) against the
real prepared sales dataset (`02_sales_data_preparation.ipynb`'s output:
35,890 rows across 32 real series from Favorita/Olist/Online Retail II),
using the already-tested code in `ml/forecasting/src/` — no training/
evaluation logic is duplicated here.

**Before running:** no GPU needed for most of this (classical/statistical
models + XGBoost run fine on CPU). A GPU runtime only helps the LSTM
candidate, which some series (≥730 days of history) will trigger.

**Disconnect-resilient by design**: results are saved to Drive after
EACH series completes, not just at the end. If Colab disconnects
partway through, re-running skips every series already saved and
resumes with whatever's left — same principle as the sentiment
notebook's checkpoint-on-Drive pattern, applied per-series here since
there's no expensive GPU training step to checkpoint instead.

Nothing here invents a metric: every MAE/RMSE/MAPE comes from
`ml/forecasting/src/metrics.py` run against each model's real
predictions on real held-out data.

In [ ]:
# Bootstrap — mounts Drive, gets the repo code, installs deps, loads the
# dataset. Idempotent: safe to call from the top of every cell below.
import os, sys, json as _json

DATA_DIR = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/sales"
REPO_DIR = "/content/insightaid-ai"
RESULTS_PATH = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/sales_forecast_results.json"  # Drive-backed, NOT /content


def bootstrap():
    global manifest, DATASET_VERSION, series_df, long_df

    try:
        import google.colab  # noqa: F401
        in_colab = True
    except ImportError:
        in_colab = False

    if in_colab and not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')

    assert os.path.exists(DATA_DIR), (
        f"Dataset folder not found: {DATA_DIR}. If your Drive layout differs, "
        f"search for it first: os.walk('/content/drive/MyDrive') looking for 'manifest.json', "
        f"then update DATA_DIR above to match."
    )

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f'git clone --depth 1 https://github.com/danishalif04/InsightAID-AI.git {REPO_DIR}')
    forecasting_src = os.path.join(REPO_DIR, "ml", "forecasting")
    if forecasting_src not in sys.path:
        sys.path.insert(0, forecasting_src)

    try:
        import statsmodels, prophet, xgboost  # noqa: F401
    except ImportError:
        get_ipython().system('pip install -q statsmodels prophet xgboost')

    if 'long_df' not in globals():
        import pandas as pd
        with open(os.path.join(DATA_DIR, "manifest.json")) as f:
            manifest = _json.load(f)
        DATASET_VERSION = manifest["dataset_version"]
        long_df = pd.read_parquet(os.path.join(DATA_DIR, "sales_daily_long.parquet"))
        series_df = pd.read_csv(os.path.join(DATA_DIR, "series_profile.csv"))
        print(f"Loaded dataset_version={DATASET_VERSION}  rows={len(long_df)}  series={series_df['series_id'].nunique()}")


bootstrap()
print("Ready. DATA_DIR =", DATA_DIR)
print("Results checkpoint file:", RESULTS_PATH, "(exists)" if os.path.exists(RESULTS_PATH) else "(not created yet)")

In [ ]:
# Build one (date, revenue) daily_df per series - the shared shape every
# model in ml/forecasting/src expects, regardless of which public dataset
# a series came from. 'y' (the prepared long-format value column) becomes
# 'revenue', matching what run_comparison()/the Prophet/XGBoost/ARIMA
# wrappers actually read.
bootstrap()
import pandas as pd

def build_daily_df_for_series(series_id):
    rows = long_df[long_df["series_id"] == series_id].sort_values("date")
    return pd.DataFrame({
        "date": pd.to_datetime(rows["date"]),
        "revenue": rows["y"].astype(float),
        "sales": rows["y"].astype(float),
    }).reset_index(drop=True)

print(f"{series_df['series_id'].nunique()} series available:")
print(series_df[["dataset", "series_id", "days"]].to_string(index=False))

In [ ]:
# Run the real model comparison for every series, saving to Drive after
# EACH one completes - a disconnect partway through costs nothing already
# saved. horizon=30 / max_folds=3 matches this dataset's own prepared
# fold convention (see docs/datasets/sales/DATASET_REPORT.md); classical
# models + XGBoost are fast, so this is CPU-only time, not GPU time.
bootstrap()
from pathlib import Path
from src.evaluate import run_comparison
from dataclasses import asdict

HORIZON = 30
MAX_FOLDS = 3

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = _json.load(f)
else:
    all_results = {}

series_ids = series_df["series_id"].tolist()
for i, series_id in enumerate(series_ids, start=1):
    if series_id in all_results:
        print(f"[{i}/{len(series_ids)}] {series_id}: already done - skipping")
        continue

    daily_df = build_daily_df_for_series(series_id)
    n_obs = len(daily_df)
    initial_train_size = max(30, n_obs - HORIZON * (MAX_FOLDS + 1))
    if initial_train_size + HORIZON > n_obs:
        print(f"[{i}/{len(series_ids)}] {series_id}: only {n_obs} days - too short for horizon={HORIZON}, skipping")
        all_results[series_id] = {"skipped": True, "reason": f"only {n_obs} observations"}
        with open(RESULTS_PATH, "w") as f:
            _json.dump(all_results, f, indent=1, default=str)
        continue

    print(f"[{i}/{len(series_ids)}] {series_id}: running real comparison ({n_obs} days)...", flush=True)
    try:
        result = run_comparison(
            daily_df, horizon=HORIZON, initial_train_size=initial_train_size,
            dataset_version=DATASET_VERSION, max_folds=MAX_FOLDS,
        )
        rows = [asdict(r) for r in result.rows]
        all_results[series_id] = {"rows": rows, "lstm_decision": result.lstm_decision}
        best = next((r for r in rows if r["is_best"]), None)
        print(f"  done - best: {best['model_type']}/{best['variant']} MAPE={best['mape']}", flush=True)
    except Exception as exc:
        print(f"  FAILED: {exc}", flush=True)
        all_results[series_id] = {"skipped": True, "reason": str(exc)}

    with open(RESULTS_PATH, "w") as f:
        _json.dump(all_results, f, indent=1, default=str)

print("\nAll series processed. Results saved to", RESULTS_PATH)

## Optional: real sentiment-enhanced comparison for Olist

Only Olist has a paired review-based signal in this dataset (see
`docs/datasets/sales/DATASET_REPORT.md`) - Favorita/Online Retail II have
no sentiment data at all, which is why `prophet_sentiment_enhanced` was
skipped for every one of their series above (correctly - see
`ml/forecasting/src/evaluate.py`'s per-candidate error handling). This
cell builds real regressors from Olist's own review data and re-runs the
comparison for `olist_total` only, to get one genuine Model A vs Model B
result.

In [ ]:
bootstrap()
OLIST_REVIEWS_PATH = os.path.join(DATA_DIR, "olist_review_text_for_sentiment.csv")

if not os.path.exists(OLIST_REVIEWS_PATH):
    print("olist_review_text_for_sentiment.csv not found at", OLIST_REVIEWS_PATH, "- skipping.")
else:
    reviews_df = pd.read_csv(OLIST_REVIEWS_PATH)
    print("Columns available:", list(reviews_df.columns))
    print(reviews_df.head(3).to_string())
    print()
    print("Inspect the columns above, then build real daily regressors matching")
    print("ml/forecasting/src/prophet_model.SENTIMENT_REGRESSOR_COLUMNS:")
    print("  avg_rating_sentiment, negative_ratio, comment_volume")
    print("This is deliberately left as a manual step - the exact column names")
    print("in this file were not confirmed before writing this notebook, and")
    print("guessing a mapping here risks silently fabricating a 'sentiment' signal")
    print("instead of using Olist's real one.")

## Record these results in Supabase

Nothing in this app writes to `forecast_results` automatically. Unlike
`model_versions` (shared, service-role-only), `forecast_results` is
per-user data with RLS on `user_id` - running this SQL from Supabase's
SQL Editor has no authenticated session, so `user_id` must be set
explicitly. **Edit `ACCOUNT_EMAIL` below to your own Supabase account
email before running this cell** - it is not filled in for you on
purpose, so it's your choice whether that value ends up in this
notebook's saved output.

In [ ]:
ACCOUNT_EMAIL = "YOUR_EMAIL_HERE"  # <-- edit this before running

def sql_literal(value):
    if value is None:
        return "NULL"
    if isinstance(value, (dict, list)):
        return "'" + _json.dumps(value).replace("'", "''") + "'::jsonb"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

with open(RESULTS_PATH) as f:
    all_results = _json.load(f)

user_id_sql = f"(select id from auth.users where email = '{ACCOUNT_EMAIL}')"
print(f"-- {sum(1 for v in all_results.values() if 'rows' in v)} series with real results, "
      f"{sum(1 for v in all_results.values() if v.get('skipped'))} skipped\n")

for series_id, entry in all_results.items():
    if "rows" not in entry:
        continue
    run_id = f"{series_id}-{DATASET_VERSION}"
    for row in entry["rows"]:
        skip_keys = {"generated_at", "run_id"}  # run_id supplied explicitly below (our own per-series id, not run_comparison()'s internal random uuid)
        cols = ["user_id", "run_id"] + [k for k in row.keys() if k not in skip_keys]
        vals = [user_id_sql, sql_literal(run_id)] + [sql_literal(row[k]) for k in row.keys() if k not in skip_keys]
        print(f"INSERT INTO forecast_results ({', '.join(cols)}) VALUES ({', '.join(vals)});")
    print()

## Save a plain-text summary too

So you have a human-readable record of which model won for each series,
not just the raw JSON/SQL above.

In [ ]:
SUMMARY_PATH = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/sales_forecast_summary.txt"

with open(RESULTS_PATH) as f:
    all_results = _json.load(f)

lines = [f"Forecast comparison summary - dataset_version={DATASET_VERSION}", ""]
for series_id, entry in all_results.items():
    if "rows" not in entry:
        lines.append(f"{series_id}: skipped ({entry.get('reason', 'unknown')})")
        continue
    best = next((r for r in entry["rows"] if r["is_best"]), None)
    models_tried = ", ".join(sorted({r["model_type"] for r in entry["rows"]}))
    lines.append(f"{series_id}: best={best['model_type']}/{best['variant']} MAPE={best['mape']} MAE={best['mae']} (tried: {models_tried})")

summary_text = "\n".join(lines)
print(summary_text)
with open(SUMMARY_PATH, "w") as f:
    f.write(summary_text)
print("\nSaved to", SUMMARY_PATH)